# Universidad del Valle de Guatemala
## Departamento de Ciencias de la Computación / Matemática
### Modelación y Simulación 2026 - Laboratorio 04

**Integrantes:**
* Erick Guerra (`eagi578@gmail.com`)
* Fabian Morales (`fabimoradav2004@gmail.com`)
* Diego Patzan (`diegopatzan24@gmail.com`)

**Fecha:** 1 al 2 de septiembre de 2026

---

## 0. Importación de Librerías y Configuración
En este notebook se utiliza la librería `PuLP` para la formulación y resolución de problemas de programación lineal (continua y entera).

In [ ]:
import pulp
import pandas as pd

print(f"Versión de PuLP cargada: {pulp.__version__}")

## Problema 2: Modelo de Producción, Períodos Múltiples (ACME Manufacturing Company)

### a) Formulación del Modelo de Programación Lineal

#### Variables de Decisión:
* $x_t \ge 0$: Cantidad de ventanas a producir en el mes $t \in \{1, 2, 3, 4, 5, 6\}$.
* $I_t \ge 0$: Cantidad de ventanas en inventario al final del mes $t \in \{1, 2, 3, 4, 5, 6\}$.

#### Función Objetivo:
Minimizar los costos totales de producción e inventario durante los 6 meses:
$$\min Z = \sum_{t=1}^{6} \left( c_t \cdot x_t + h_t \cdot I_t \right)$$

donde los costos unitarios de producción son $c = [50, 45, 55, 52, 48, 50]$ y los costos unitarios de almacenamiento son $h = [8, 10, 10, 10, 8, 8]$.

#### Restricciones:
1. **Balance de Inventario por Período ($I_{t} = I_{t-1} + x_t - d_t$ con $I_0 = 0$):**
   * Mes 1: $I_1 = x_1 - 180$
   * Mes 2: $I_2 = I_1 + x_2 - 250$
   * Mes 3: $I_3 = I_2 + x_3 - 190$
   * Mes 4: $I_4 = I_3 + x_4 - 140$
   * Mes 5: $I_5 = I_4 + x_5 - 220$
   * Mes 6: $I_6 = I_5 + x_6 - 250$
2. **Capacidad Máxima de Producción Mensual:**
   $$x_t \le 225, \quad \forall t \in \{1..6\}$$
3. **No Negatividad:**
   $$x_t \ge 0, \quad I_t \ge 0, \quad \forall t \in \{1..6\}$$

In [ ]:
meses = list(range(1, 7))
demanda = {1: 180, 2: 250, 3: 190, 4: 140, 5: 220, 6: 250}
costo_prod = {1: 50, 2: 45, 3: 55, 4: 52, 5: 48, 6: 50}
costo_inv = {1: 8, 2: 10, 3: 10, 4: 10, 5: 8, 6: 8}
capacidad_max = 225

# Modelo Continuo
prob_cont = pulp.LpProblem("ACME_Produccion_Continuo", pulp.LpMinimize)
x_cont = pulp.LpVariable.dicts("Prod", meses, lowBound=0, cat=pulp.LpContinuous)
I_cont = pulp.LpVariable.dicts("Inv", meses, lowBound=0, cat=pulp.LpContinuous)

prob_cont += pulp.lpSum([costo_prod[t] * x_cont[t] + costo_inv[t] * I_cont[t] for t in meses])

for t in meses:
    prob_cont += x_cont[t] <= capacidad_max, f"Capacidad_Mes_{t}"
    if t == 1:
        prob_cont += I_cont[t] == x_cont[t] - demanda[t], f"Balance_Mes_{t}"
    else:
        prob_cont += I_cont[t] == I_cont[t-1] + x_cont[t] - demanda[t], f"Balance_Mes_{t}"

prob_cont.solve(pulp.PULP_CBC_CMD(msg=0))

# Modelo Entero
prob_ent = pulp.LpProblem("ACME_Produccion_Entero", pulp.LpMinimize)
x_ent = pulp.LpVariable.dicts("Prod", meses, lowBound=0, cat=pulp.LpInteger)
I_ent = pulp.LpVariable.dicts("Inv", meses, lowBound=0, cat=pulp.LpInteger)

prob_ent += pulp.lpSum([costo_prod[t] * x_ent[t] + costo_inv[t] * I_ent[t] for t in meses])

for t in meses:
    prob_ent += x_ent[t] <= capacidad_max, f"Capacidad_Mes_{t}"
    if t == 1:
        prob_ent += I_ent[t] == x_ent[t] - demanda[t], f"Balance_Mes_{t}"
    else:
        prob_ent += I_ent[t] == I_ent[t-1] + x_ent[t] - demanda[t], f"Balance_Mes_{t}"

prob_ent.solve(pulp.PULP_CBC_CMD(msg=0))

# Presentación de Resultados en Tabla
df_res2 = pd.DataFrame({
    'Mes': meses,
    'Demanda': [demanda[t] for t in meses],
    'Producción (x_t)': [x_cont[t].varValue for t in meses],
    'Inventario Final (I_t)': [I_cont[t].varValue for t in meses],
    'Costo Prod ($)': [x_cont[t].varValue * costo_prod[t] for t in meses],
    'Costo Inv ($)': [I_cont[t].varValue * costo_inv[t] for t in meses]
})
df_res2['Costo Total ($)'] = df_res2['Costo Prod ($)'] + df_res2['Costo Inv ($)']

print("==========================================================")
print("PROBLEMA 2: MODELO DE PRODUCCIÓN E INVENTARIO (ACME)")
print("==========================================================")
print(f"Estado Solución: {pulp.LpStatus[prob_cont.status]}")
print(f"Costo Total Óptimo: ${pulp.value(prob_cont.objective):,.2f}\n")
print(df_res2.to_string(index=False))

print("\n--- DIAGRAMA DE FLUJO DE PRODUCCIÓN E INVENTARIO ---")
print("I0 = 0")
for t in meses:
    prev_inv = 0 if t == 1 else I_cont[t-1].varValue
    print(f"Mes {t}: Inv_Inic ({prev_inv:.0f}) + Prod ({x_cont[t].varValue:.0f}) - Demanda ({demanda[t]}) ==> Inv_Fin ({I_cont[t].varValue:.0f})")

print("\n--- COMPARACIÓN CON MODELO ENTERO (Inciso c) ---")
print(f"Costo Óptimo Modelo Entero: ${pulp.value(prob_ent.objective):,.2f}")
print("¿Se obtiene la misma solución óptima? SÍ, la solución continua resulta ser naturally entera.")